# 01. Khám phá dữ liệu Student Performance

Notebook này giúp nhóm hiểu dữ liệu trước khi xây dựng mô hình. Đề tài chính là dự đoán điểm cuối kỳ `G3` của học sinh.

Nội dung gồm: kiểm tra kích thước và kiểu dữ liệu, dữ liệu thiếu, dữ liệu trùng, phân bố điểm, mối quan hệ giữa các biến và tối thiểu bốn biểu đồ EDA.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_project_root():
    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / 'data').exists() and (folder / 'src').exists():
            return folder
    raise FileNotFoundError('Không tìm thấy thư mục gốc của project.')

PROJECT_ROOT = find_project_root()
MERGED_FILE = PROJECT_ROOT / 'data' / 'interim' / 'merged.csv'
FIGURES_DIR = PROJECT_ROOT / 'artifacts' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 40)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Đọc merged.csv. Nếu file trung gian chưa có, đọc trực tiếp hai file raw.
if MERGED_FILE.exists():
    df = pd.read_csv(MERGED_FILE)
else:
    raw_dir = PROJECT_ROOT / 'data' / 'raw'
    math_df = pd.read_csv(raw_dir / 'student-mat.csv', sep=';')
    math_df.insert(0, 'subject', 'math')
    portuguese_df = pd.read_csv(raw_dir / 'student-por.csv', sep=';')
    portuguese_df.insert(0, 'subject', 'portuguese')
    df = pd.concat([math_df, portuguese_df], ignore_index=True)

print(f'Kích thước dữ liệu: {df.shape[0]} dòng x {df.shape[1]} cột')
display(df.head())
display(df.dtypes.rename('dtype').to_frame())

In [ ]:
# Kiểm tra chất lượng dữ liệu trước khi phân tích.
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('Các cột có giá trị thiếu:')
display(missing.to_frame('missing_count') if not missing.empty else pd.DataFrame({'missing_count': [0]}, index=['Không có']))
print(f'Số dòng trùng hoàn toàn: {df.duplicated().sum()}')
print('Số dòng theo môn học:')
display(df['subject'].value_counts().rename('count').to_frame())

numeric_columns = df.select_dtypes(include='number').columns.tolist()
display(df[numeric_columns].describe().T)

In [ ]:
# Thống kê mục tiêu G3 theo môn học.
subject_summary = df.groupby('subject')['G3'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)
display(subject_summary)

pass_rate = (df['G3'] >= 10).groupby(df['subject']).mean().mul(100).round(2)
print('Tỷ lệ đạt nếu dùng ngưỡng G3 >= 10:')
display(pass_rate.rename('pass_rate_percent').to_frame())

print('Trung bình G3 theo thời gian học:')
display(df.groupby('studytime')['G3'].mean().round(2).rename('mean_G3').to_frame())

In [ ]:
# Biểu đồ 1: phân bố điểm G3 theo môn học.
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=df, x='G3', hue='subject', discrete=True, multiple='dodge', shrink=0.8, ax=ax)
ax.set_title('Phân bố điểm cuối kỳ G3 theo môn học')
ax.set_xlabel('Điểm G3')
ax.set_ylabel('Số học sinh')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_g3_distribution.png', dpi=150)
plt.show()

In [ ]:
# Biểu đồ 2: so sánh điểm G3 giữa hai môn.
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x='subject', y='G3', ax=ax)
ax.set_title('So sánh phân bố điểm G3 giữa hai môn')
ax.set_xlabel('Môn học')
ax.set_ylabel('Điểm G3')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_g3_by_subject.png', dpi=150)
plt.show()

In [ ]:
# Biểu đồ 3: điểm các kỳ trước và điểm cuối kỳ.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df, x='G1', y='G3', hue='subject', alpha=0.65, ax=axes[0])
axes[0].set_title('Mối quan hệ giữa G1 và G3')
sns.scatterplot(data=df, x='G2', y='G3', hue='subject', alpha=0.65, ax=axes[1])
axes[1].set_title('Mối quan hệ giữa G2 và G3')
for ax in axes:
    ax.set_xlabel('Điểm kỳ trước')
    ax.set_ylabel('Điểm G3')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_previous_grades_vs_g3.png', dpi=150)
plt.show()

In [ ]:
# Biểu đồ 4: ma trận tương quan của các biến số.
corr = df.select_dtypes(include='number').corr()
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Ma trận tương quan các biến số')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_numeric_correlation.png', dpi=150)
plt.show()

print('Tương quan với G3:')
display(corr['G3'].sort_values(ascending=False).to_frame('correlation_with_G3'))

In [ ]:
# Biểu đồ 5: số lần vắng học và điểm G3.
absence_bins = [-1, 5, 10, 20, 100]
absence_labels = ['0-5', '6-10', '11-20', '21+']
plot_df = df.assign(absence_group=pd.cut(df['absences'], bins=absence_bins, labels=absence_labels))
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=plot_df, x='absence_group', y='G3', ax=ax)
ax.set_title('Điểm G3 theo nhóm số lần vắng học')
ax.set_xlabel('Số lần vắng học')
ax.set_ylabel('Điểm G3')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_absences_vs_g3.png', dpi=150)
plt.show()

In [ ]:
# Lưu một số bảng tổng hợp để dùng lại trong báo cáo.
summary = {
    'rows': int(df.shape[0]),
    'columns': int(df.shape[1]),
    'subjects': {str(k): int(v) for k, v in df['subject'].value_counts().items()},
    'g3_mean': float(df['G3'].mean()),
    'g3_median': float(df['G3'].median()),
    'missing_values': int(df.isna().sum().sum()),
    'duplicate_rows': int(df.duplicated().sum())
}
(PROJECT_ROOT / 'artifacts' / 'eda_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Đã lưu bảng và biểu đồ EDA vào artifacts/.')

## Kết luận cần ghi vào báo cáo

Sau khi chạy notebook, nhóm ghi lại: dữ liệu có bao nhiêu dòng/cột, có dữ liệu thiếu hay không, phân bố `G3` giữa hai môn, mối liên hệ của `G1`, `G2`, `absences` với `G3`, và những biến sẽ được đưa vào mô hình. Không nên kết luận nguyên nhân nhân quả chỉ từ tương quan.